# ViralCutter
Uma alternativa gratuita ao `opus.pro` e ao `vidyo.ai`

# Suporte em:
[![](https://dcbadge.limes.pink/api/server/tAdPHFAbud)](https://discord.gg/tAdPHFAbud)

# TODO📝
- [x] Release code
- [ ] Huggingface SpaceDemo
- [x] Two face in the cut
- [x] Custom caption and burn
- [x] Make the code faster
- [ ] More types of framing beyond 9:16

In [ ]:
#@title 🛠️ ViralCutter Installation
import os
import shutil
import subprocess
import sys
from pathlib import Path

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

REPO_URL = "https://github.com/kodokbakar/ViralCutter.git"

# Change this when the feature branch has been merged into main.
REPO_BRANCH = "main"

PROJECT_DIR = Path("/content/ViralCutter")
VENV_DIR = PROJECT_DIR / ".venv"
VENV_PYTHON = VENV_DIR / "bin" / "python"
INSTALL_LOG = Path("/content/viralcutter_install.log")


# ---------------------------------------------------------------------
# Command runner
# ---------------------------------------------------------------------

def run_step(title, command, cwd=None, check=True, env=None):
    print()
    print("=" * 80)
    print(f"▶ {title}")
    print("=" * 80)

    command = [str(part) for part in command]
    print("$", " ".join(command))
    print()

    process_env = os.environ.copy()
    if env:
        process_env.update(env)

    with INSTALL_LOG.open("a", encoding="utf-8") as log_file:
        log_file.write("\n" + "=" * 80 + "\n")
        log_file.write(f"{title}\n")
        log_file.write("$ " + " ".join(command) + "\n")
        log_file.flush()

        process = subprocess.Popen(
            command,
            cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=process_env,
        )

        output_lines = []

        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()
            output_lines.append(line)

        return_code = process.wait()

    if return_code != 0:
        message = (
            f"\n❌ Step failed: {title}\n"
            f"Exit code: {return_code}\n"
            f"Full log: {INSTALL_LOG}\n"
        )

        print(message)

        if check:
            raise RuntimeError(message)

    print(f"\n✅ Completed: {title}")
    return return_code


def uv_install(*packages, extra_args=None, title=None):
    command = [
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_DIR),
    ]

    if extra_args:
        command.extend(extra_args)

    command.extend(packages)

    return run_step(
        title or f"Installing {' '.join(packages)}",
        command,
        cwd=PROJECT_DIR,
    )


# ---------------------------------------------------------------------
# 1. Clean previous installation
# ---------------------------------------------------------------------

print("🧹 Cleaning previous installation...")

os.chdir("/content")
INSTALL_LOG.unlink(missing_ok=True)

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)


# ---------------------------------------------------------------------
# 2. Clone repository
# ---------------------------------------------------------------------

clone_command = [
    "git",
    "clone",
    "--single-branch",
    "--branch",
    REPO_BRANCH,
    REPO_URL,
    str(PROJECT_DIR),
]

run_step(
    "Cloning ViralCutter",
    clone_command,
    cwd="/content",
)

os.chdir(PROJECT_DIR)

run_step(
    "Showing repository revision",
    ["git", "log", "-1", "--oneline", "--decorate"],
    cwd=PROJECT_DIR,
)


# ---------------------------------------------------------------------
# 3. Install UV and system dependencies
# ---------------------------------------------------------------------

run_step(
    "Installing UV",
    [sys.executable, "-m", "pip", "install", "--upgrade", "uv"],
)

run_step(
    "Installing system packages",
    [
        "sudo",
        "apt-get",
        "install",
        "-y",
        "ffmpeg",
        "xvfb",
        "libcudnn8",
    ],
)


# ---------------------------------------------------------------------
# 4. Create virtual environment
# ---------------------------------------------------------------------

run_step(
    "Creating virtual environment",
    [
        "uv",
        "venv",
        str(VENV_DIR),
        "--python",
        sys.executable,
    ],
    cwd=PROJECT_DIR,
)

run_step(
    "Checking virtual-environment Python",
    [str(VENV_PYTHON), "--version"],
)


# ---------------------------------------------------------------------
# 5. Install stable CUDA Torch stack first
# ---------------------------------------------------------------------

uv_install(
    "torch==2.3.1+cu121",
    "torchvision==0.18.1+cu121",
    "torchaudio==2.3.1+cu121",
    extra_args=[
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
    ],
    title="Installing stable PyTorch CUDA 12.1 stack",
)


# ---------------------------------------------------------------------
# 6. Install the compatible WhisperX stack
# ---------------------------------------------------------------------

# Do not install the current GitHub main branch here.
# WhisperX 3.2.0 matches CTranslate2 4.4.0 and faster-whisper 1.0.0.

uv_install(
    "whisperx==3.2.0",
    "ctranslate2==4.4.0",
    "faster-whisper==1.0.0",
    title="Installing compatible WhisperX stack",
)


# ---------------------------------------------------------------------
# 7. Install project requirements
# ---------------------------------------------------------------------

requirements_file = PROJECT_DIR / "requirements-colab.txt"

if not requirements_file.exists():
    raise FileNotFoundError(
        f"Missing requirements file: {requirements_file}"
    )

uv_install(
    "-r",
    str(requirements_file),
    title="Installing ViralCutter Colab requirements",
)


# ---------------------------------------------------------------------
# 8. Pin compatibility-sensitive libraries
# ---------------------------------------------------------------------

uv_install(
    "numpy<2.0",
    "setuptools==69.5.1",
    "transformers==4.46.3",
    "accelerate>=0.26.0",
    title="Applying NumPy and Transformers compatibility pins",
)


# ---------------------------------------------------------------------
# 9. Install G4F separately
# ---------------------------------------------------------------------

# Start with the smaller official optional group.
# A failure here should be easy to identify and will not be confused
# with WhisperX installation failures.

g4f_status = run_step(
    "Installing G4F slim backend",
    [
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_DIR),
        "g4f[slim]",
    ],
    cwd=PROJECT_DIR,
    check=False,
)

if g4f_status != 0:
    print()
    print("⚠️ g4f[slim] failed. Trying the base G4F package.")

    run_step(
        "Installing base G4F package",
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(VENV_DIR),
            "g4f",
        ],
        cwd=PROJECT_DIR,
        check=False,
    )


# ---------------------------------------------------------------------
# 10. Install computer-vision dependencies
# ---------------------------------------------------------------------

uv_install(
    "insightface",
    "onnxruntime-gpu",
    title="Installing InsightFace and ONNX Runtime GPU",
)

run_step(
    "Removing conflicting MediaPipe packages",
    [
        "uv",
        "pip",
        "uninstall",
        "--python",
        str(VENV_DIR),
        "mediapipe",
        "protobuf",
        "flatbuffers",
    ],
    cwd=PROJECT_DIR,
    check=False,
)

uv_install(
    "mediapipe>=0.10.0",
    "protobuf>=3.20,<5.0",
    "flatbuffers>=2.0",
    title="Installing MediaPipe compatibility stack",
)


# ---------------------------------------------------------------------
# 11. Verify imports and versions
# ---------------------------------------------------------------------

verification_code = r"""
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

import numpy
print("NumPy:", numpy.__version__)

import torch
print("Torch:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import ctranslate2
print("CTranslate2:", ctranslate2.__version__)

import faster_whisper
print("faster-whisper: OK")

import whisperx
print("WhisperX:", getattr(whisperx, "__version__", "version attribute unavailable"))

import transformers
print("Transformers:", transformers.__version__)

import gradio
print("Gradio:", gradio.__version__)

import googleapiclient
import google.auth
import google_auth_oauthlib
print("Google Drive API libraries: OK")

try:
    import g4f
    print("G4F:", getattr(g4f, "__version__", "installed"))
except Exception as exc:
    print("G4F unavailable:", exc)

try:
    import insightface
    print("InsightFace:", getattr(insightface, "__version__", "installed"))
except Exception as exc:
    print("InsightFace unavailable:", exc)

try:
    import mediapipe
    print("MediaPipe:", mediapipe.__version__)
except Exception as exc:
    print("MediaPipe unavailable:", exc)
"""

run_step(
    "Verifying ViralCutter runtime",
    [
        str(VENV_PYTHON),
        "-u",
        "-c",
        verification_code,
    ],
    cwd=PROJECT_DIR,
)


# ---------------------------------------------------------------------
# 12. Advisory dependency check
# ---------------------------------------------------------------------

print()
print("🔍 Checking dependency consistency...")

dependency_check = subprocess.run(
    [
        "uv",
        "pip",
        "check",
        "--python",
        str(VENV_DIR),
    ],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)

if dependency_check.stdout:
    print(dependency_check.stdout)

if dependency_check.stderr:
    print(dependency_check.stderr)

if dependency_check.returncode != 0:
    print(
        "⚠️ Dependency check reported issues. "
        "Review them before starting a long transcription job."
    )
else:
    print("✅ All installed dependencies are consistent.")


# ---------------------------------------------------------------------
# 13. Start virtual display
# ---------------------------------------------------------------------

existing_xvfb = subprocess.run(
    ["pgrep", "-f", "Xvfb :1"],
    capture_output=True,
    text=True,
)

if existing_xvfb.returncode != 0:
    subprocess.Popen(
        [
            "Xvfb",
            ":1",
            "-screen",
            "0",
            "2560x1440x24",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

os.environ["DISPLAY"] = ":1.0"
os.environ["MPLBACKEND"] = "Agg"


# ---------------------------------------------------------------------
# Result
# ---------------------------------------------------------------------

print()
print("=" * 80)
print("✅ ViralCutter installation completed")
print("=" * 80)
print(f"Project: {PROJECT_DIR}")
print(f"Python: {VENV_PYTHON}")
print(f"Install log: {INSTALL_LOG}")

✅ Instalação V7 Finalizada!
- Transformers 4.46.3 (Compatível com Alinhamento): INSTALADO
- Torch 2.3.1: ATIVO


In [ ]:
#@title 🩺 Runtime Doctor Preflight
import subprocess

%cd /content/ViralCutter

print("🩺 Running ViralCutter runtime doctor...")
subprocess.run(
    [
        "/content/ViralCutter/.venv/bin/python",
        "-c",
        "import sys; sys.path.insert(0, '/content/ViralCutter/webui'); import runtime_doctor; print(runtime_doctor.run_runtime_doctor())",
    ],
    check=False,
)

In [ ]:
#@title 🚀 Configuration and Run
import os
import subprocess
from google.colab import drive

%cd /content/ViralCutter

print("🔐 Mounting Google Drive...")
drive.mount("/content/drive")
drive_output_dir = "/content/drive/MyDrive/ViralCutter/VIRALS"
os.makedirs(drive_output_dir, exist_ok=True)

os.environ["VIRALCUTTER_OUTPUT_DIR"] = drive_output_dir

print(f"📁 ViralCutter outputs will be saved to: {drive_output_dir}")

os.system("Xvfb :1 -screen 0 2560x1440x8 &")
os.environ["DISPLAY"] = ":1.0"
os.environ["MPLBACKEND"] = "Agg"

print("🚀 Starting ViralCutter with the virtual environment...")
print("⚠️ Ignore harmless UserWarning messages. Wait for the public URL.")

!/content/ViralCutter/.venv/bin/python webui/app.py --colab

/content/ViralCutter
🚀 Iniciando ViralCutter usando o ambiente virtual...
⚠️ Ignore os avisos de 'UserWarning', aguarde o link public URL.
/content/ViralCutter/webui/app.py:204: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title=i18n("ViralCutter WebUI"), theme=gr.themes.Default(primary_hue="blue", neutral_hue="slate"), css=css) as demo:
Running in Colab mode. Generating public link...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7f3b04ec670707105b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#Créditos

Inspirado no [reels clips automator](https://github.com/eddieoz/reels-clips-automator) e no [YoutubeVideoToAIPoweredShorts](https://github.com/Fitsbit/YoutubeVideoToAIPoweredShorts)<br>

---
![Rafa.png](https://i.imgur.com/cGknQpU.png;base64)

Desenvolvido por **Rafa.Godoy**<br>
[ ![GitHub](https://img.shields.io/badge/github-%23121011.svg?style=for-the-badge&logo=github&logoColor=white) ](https://github.com/rafaelGodoyEbert)<br>
[ ![X](https://img.shields.io/twitter/url?url=https%3A%2F%2Ftwitter.com%2FGodoyEbert) ](https://twitter.com/GodoyEbert)<br>
[Instagram](https://www.instagram.com/rafael.godoy.ebert/)<br>
[ ![](https://dcbadge.vercel.app/api/server/aihubbrasil) ](https://discord.gg/aihubbrasil)

`0.1v Alpha`<br>

Apenas uma alternativa gratuita ao `opus.pro` e ao `vidyo.ai`<br>
